# 05d - Deep Learning Heavy

Este notebook executa um experimento PyTorch mais robusto sobre sinais brutos `records500`. O conjunto de teste ? usado apenas uma vez ao final, ap?s sele??o por valida??o no fold 9.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))


## Ambiente e configura??o

O c?digo usa CUDA automaticamente quando dispon?vel. Em CPU, a configura??o limita o n?mero de execu??es para manter o custo computacional controlado.

In [2]:
from tcc_ecg.config import load_config
from tcc_ecg.data import prepare_metadata
from tcc_ecg.dl_training import get_torch_environment, get_deep_learning_heavy_config

config = load_config()
env = get_torch_environment()
heavy_config = get_deep_learning_heavy_config(config)
print(env)
print({k: heavy_config[k] for k in ['frequency', 'model_family', 'architectures', 'max_runs', 'batch_size', 'epochs', 'patience', 'use_focal_loss', 'use_weighted_sampler']})

{'python': '3.11.9', 'torch': '2.12.0+cpu', 'cuda_available': False, 'device': 'CPU', 'gpu_memory_bytes': 0}
{'frequency': 500, 'model_family': 'inceptiontime_resnet_ensemble', 'architectures': ['resnet1d_se', 'inceptiontime_deep'], 'max_runs': 1, 'batch_size': 64, 'epochs': 120, 'patience': 20, 'use_focal_loss': True, 'use_weighted_sampler': True}


## Prepara??o dos metadados

A constru??o dos r?tulos mant?m a estrat?gia `strict_single_label`. Os folds oficiais s?o preservados: treino 1-8, valida??o 9 e teste 10.

In [3]:
metadata = prepare_metadata(config, save_summary=True)
metadata[['target', 'target_id', 'strat_fold']].dropna().head()

,target,target_id,strat_fold
ecg_id,,,
1,NORM,0,3
2,NORM,0,2
3,NORM,0,5
4,NORM,0,3
5,NORM,0,4


## Treinamento e avalia??o

O treinamento usa normaliza??o por canal calculada apenas no treino, class weights calculados apenas no treino, WeightedRandomSampler somente no DataLoader de treino e aumenta??es leves apenas no treino.

In [4]:
from pathlib import Path
import pandas as pd
from tcc_ecg.dl_training import train_deep_learning_heavy

metrics_path = PROJECT_ROOT / 'reports' / 'tables' / 'deep_learning_heavy_metrics.csv'
if metrics_path.exists():
    result = {'metrics': pd.read_csv(metrics_path).iloc[0].to_dict(), 'skipped_training': True}
else:
    result = train_deep_learning_heavy(metadata, config)
result['metrics']

{'accuracy': 0.7248484848484849,
 'balanced_accuracy': 0.6677156166332107,
 'precision_macro': 0.6202433348439516,
 'recall_macro': 0.6677156166332107,
 'f1_macro': 0.6389955729431165,
 'f1_weighted': 0.7312211099879027,
 'model': 'deep_learning_heavy_resnet1d_se',
 'split': 'test',
 'signal_frequency': 500,
 'smote': False,
 'device': 'cpu',
 'best_epoch': 71,
 'epochs_trained': 91,
 'training_seconds': 49759.88336110115,
 'n_parameters': 1309149,
 'architecture': 'resnet1d_se',
 'loss_type': 'focal'}

## Artefatos gerados

- `models/deep_learning_heavy_best.pt`;
- `reports/tables/deep_learning_heavy_metrics.csv` e `.tex`;
- `reports/tables/deep_learning_heavy_classification_report.csv` e `.tex`;
- `reports/tables/deep_learning_heavy_history.csv`;
- figuras de curvas de treino e matriz de confus?o;
- `reports/tables/deep_learning_comparison.csv` e `.tex`.